# Flood depth from roadside cameras: fine-tune on a Colab T4

1. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
2. When the **Choose Files** button appears below, pick `flood_camera_colab.zip` (made by `python -m src.pipeline.sunnyday --pack`).
3. Leave the tab open. The last cell downloads `flood_camera_results.zip`.

The zip holds the frames, the depth labels and `src/model/flood_camera.py`; this notebook only runs it.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.'
print(torch.cuda.get_device_name(0))

In [ ]:
import glob
from google.colab import files
if not glob.glob('*.zip'):
    files.upload()  # choose flood_camera_colab.zip
package = sorted(glob.glob('flood_camera_colab*.zip'))[-1]
!unzip -q -o "{package}"
!pip -q install timm pyarrow
!ls sunnyday | head

In [ ]:
EPOCHS = 6
# cameras the model has never seen: one camera site held out at a time; then one model on every frame,
# scored on the NCDOT stills
!python -m src.model.flood_camera --split camera --epochs {EPOCHS} --final --data sunnyday --out out

In [ ]:
# cameras it has seen, on a day it has not
!python -m src.model.flood_camera --split day --epochs {EPOCHS} --data sunnyday --out out

In [ ]:
!cd out && zip -q -r ../flood_camera_results.zip .
files.download('flood_camera_results.zip')